# Docker Essentials for ML Practitioners

Docker solves one of the most frustrating problems in machine learning: the gap between the environment where code was written and the environment where it runs. This notebook walks through the core concepts, shows you how to write Dockerfiles for GPU-accelerated training, and ends with a hands-on exercise containerizing a small PyTorch script.

> **Note:** Cells prefixed with `!` run shell commands and require Docker to be installed on your machine. The `%%writefile` cells only write files to disk and will work anywhere.

## 1. Images vs. Containers

The two words that confuse everyone at first:

**An image** is a snapshot. Think of it as a recipe or a frozen template: it specifies an operating system, installed libraries, environment variables, and the default command to run. Images are immutable. You build them once and they don't change.

**A container** is a running instance of an image. You can spin up many containers from the same image, and each one is isolated from the others. When a container stops, any changes it made to its own filesystem are discarded (unless you explicitly mount a volume).

The analogy that usually sticks: an image is a class definition in Python, and a container is an object (an instance) of that class. You can instantiate as many objects as you want from the same class.

```
Dockerfile  -->  docker build  -->  Image  -->  docker run  -->  Container(s)
```

For ML work, you typically:
1. Choose a base image (e.g., `nvidia/cuda:12.1.0-cudnn8-runtime-ubuntu22.04`)
2. Add your Python dependencies on top
3. Copy your training code in
4. Run the container, passing in data via a mounted volume

## 2. Reading and Writing a Dockerfile

A Dockerfile is a plain text file with a series of instructions. Each instruction adds a layer to the image. Here are the ones you'll use most:

| Instruction | Purpose |
|---|---|
| `FROM` | Sets the base image. Every Dockerfile starts with this. |
| `RUN` | Executes a shell command at build time (e.g., install packages). |
| `COPY` | Copies files from your local machine into the image. |
| `WORKDIR` | Sets the working directory inside the container. |
| `ENV` | Sets an environment variable. |
| `CMD` | Default command to run when the container starts. |

Let's write a real Dockerfile for a PyTorch environment.

In [ ]:
%%writefile Dockerfile.pytorch
# Start from an official CUDA base image.
# This gives us CUDA 12.1, cuDNN 8, on Ubuntu 22.04.
FROM nvidia/cuda:12.1.0-cudnn8-runtime-ubuntu22.04

# Prevent interactive prompts during apt installs
ENV DEBIAN_FRONTEND=noninteractive

# Install Python and pip
RUN apt-get update && apt-get install -y \
    python3.10 \
    python3-pip \
    && rm -rf /var/lib/apt/lists/*

# Set working directory
WORKDIR /workspace

# Install Python dependencies.
# Copy requirements first so Docker can cache this layer.
# If requirements.txt doesn't change, this layer is reused on rebuild.
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy your training script
COPY train.py .

# Default command: run the training script
CMD ["python3", "train.py"]

In [ ]:
%%writefile requirements.txt
torch==2.2.0
torchvision==0.17.0
numpy==1.26.4

### Layer Caching

One detail worth understanding early: Docker builds images layer by layer, and it caches each layer. If you change a line in the Dockerfile, Docker rebuilds from that line onward.

This is why `COPY requirements.txt .` and `RUN pip install` come **before** `COPY train.py .`. Your Python dependencies change rarely; your training script changes often. By ordering things this way, you avoid re-running `pip install` every time you edit a single line of training code.

## 3. Running a Container with GPU Passthrough

By default, containers are isolated from the host system's hardware, including the GPU. To give a container access to your NVIDIA GPU, you need two things:

1. **NVIDIA Container Toolkit** installed on your host machine (`nvidia-container-toolkit`)
2. The `--gpus` flag when running `docker run`

The most common options:

```bash
# Give the container access to all GPUs
docker run --gpus all my-image

# Give access to GPU 0 only
docker run --gpus device=0 my-image

# Give access to the first two GPUs
docker run --gpus '"device=0,1"' my-image
```

You can verify GPU access inside the container by running `nvidia-smi`:

In [ ]:
# Check if Docker can see the GPU
# This requires Docker + NVIDIA Container Toolkit to be installed
!docker run --rm --gpus all nvidia/cuda:12.1.0-base-ubuntu22.04 nvidia-smi

### Mounting Data with Volumes

Since containers don't persist data by default, you mount a directory from your host machine into the container using `-v`:

```bash
docker run --rm --gpus all \
  -v /path/to/your/data:/workspace/data \
  -v /path/to/save/checkpoints:/workspace/checkpoints \
  my-training-image
```

The format is `-v host_path:container_path`. Your training script then reads from `/workspace/data` and writes to `/workspace/checkpoints`, which map to real directories on your host.

## 4. CUDA and Driver Compatibility

This is where ML practitioners get burned most often. There are three version numbers that must be compatible:

1. **NVIDIA driver version** (on your host machine, run `nvidia-smi` to check)
2. **CUDA version** (in the Docker base image, e.g., `cuda:12.1`)
3. **PyTorch version** (must be built against a compatible CUDA version)

The rule: your **host driver** determines the **maximum CUDA version** that can run inside any container on that machine. The CUDA version inside the container must be less than or equal to what your driver supports.

| NVIDIA Driver Version | Max Supported CUDA |
|---|---|
| >= 525.60 | CUDA 12.x |
| >= 450.80 | CUDA 11.x |
| >= 390.00 | CUDA 9.x |

### Common Mismatch Errors

**Error:** `CUDA error: no kernel image is available for execution on the device`  
**Cause:** PyTorch was built for a different GPU architecture (compute capability) than what you have.  
**Fix:** Install the PyTorch wheel that matches your CUDA version: `pip install torch --index-url https://download.pytorch.org/whl/cu121`

**Error:** `libcuda.so.1: cannot open shared object file`  
**Cause:** The CUDA runtime in the container can't find the driver library.  
**Fix:** Make sure `--gpus all` is passed and NVIDIA Container Toolkit is installed.

**Error:** `CUDA driver version is insufficient for CUDA runtime version`  
**Cause:** Your base image uses CUDA 12.x but your host driver only supports CUDA 11.x.  
**Fix:** Use a base image with a lower CUDA version, or update your host driver.

In [ ]:
import subprocess

def check_cuda_compatibility():
    """Print host NVIDIA driver info to help verify compatibility."""
    try:
        result = subprocess.run(
            ["nvidia-smi", "--query-gpu=driver_version,name,memory.total",
             "--format=csv,noheader"],
            capture_output=True, text=True, check=True
        )
        print("GPU Info (driver, name, VRAM):")
        for line in result.stdout.strip().split("\n"):
            print(" ", line)
    except FileNotFoundError:
        print("nvidia-smi not found. Are you on a machine without an NVIDIA GPU?")
    except subprocess.CalledProcessError as e:
        print(f"nvidia-smi failed: {e.stderr}")

check_cuda_compatibility()

## 5. Why Reproducibility Matters

"It works on my machine" is a joke, but it's also a real problem that costs real time.

Consider what can silently differ between two machines:
- Python version (3.9 vs 3.10 vs 3.11)
- Library versions (`transformers==4.35` vs `4.40` has changed defaults)
- CUDA version (affects numerical results in some ops)
- OS-level libraries (`libsndfile`, `ffmpeg` for audio)
- Environment variables that change behavior

In ML specifically, version mismatches can produce models that train but produce subtly wrong outputs, or training runs that fail silently with wrong results rather than crashing loudly.

Docker addresses this by packaging the entire runtime environment. The image you build on your laptop and push to a registry is the exact same environment that runs on the hackathon evaluation server. No surprises.

The workflow:
```bash
# Build the image
docker build -t my-experiment:v1 -f Dockerfile.pytorch .

# Test locally
docker run --rm --gpus all my-experiment:v1

# Push to a registry so teammates or servers can use it
docker tag my-experiment:v1 registry.example.com/myteam/my-experiment:v1
docker push registry.example.com/myteam/my-experiment:v1
```

## 6. Hands-On: Containerize a PyTorch Training Script

We'll build a container that:
1. Installs PyTorch
2. Runs a small training script (a 2-layer MLP on synthetic data)
3. Saves the trained model to a mounted volume

First, let's write the training script.

In [ ]:
%%writefile train.py
"""A minimal PyTorch training script for container testing."""
import os
import torch
import torch.nn as nn

# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# --- Synthetic dataset ---
torch.manual_seed(42)
X = torch.randn(1000, 16).to(device)
# Binary classification: label = 1 if sum of first 8 features > 0
y = (X[:, :8].sum(dim=1) > 0).float().to(device)

# --- Model ---
model = nn.Sequential(
    nn.Linear(16, 64),
    nn.ReLU(),
    nn.Linear(64, 1),
    nn.Sigmoid(),
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCELoss()

# --- Training loop ---
EPOCHS = 20
for epoch in range(EPOCHS):
    model.train()
    optimizer.zero_grad()
    preds = model(X).squeeze()
    loss = criterion(preds, y)
    loss.backward()
    optimizer.step()

    acc = ((preds > 0.5).float() == y).float().mean().item()
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:02d}/{EPOCHS} | loss={loss.item():.4f} | acc={acc:.3f}")

# --- Save model ---
out_dir = "/workspace/output"
os.makedirs(out_dir, exist_ok=True)
save_path = os.path.join(out_dir, "model.pt")
torch.save(model.state_dict(), save_path)
print(f"Model saved to {save_path}")

In [ ]:
%%writefile Dockerfile
FROM nvidia/cuda:12.1.0-cudnn8-runtime-ubuntu22.04

ENV DEBIAN_FRONTEND=noninteractive

RUN apt-get update && apt-get install -y \
    python3.10 \
    python3-pip \
    && rm -rf /var/lib/apt/lists/*

WORKDIR /workspace

# Install PyTorch for CUDA 12.1
RUN pip install --no-cache-dir \
    torch==2.2.0 \
    --index-url https://download.pytorch.org/whl/cu121

COPY train.py .

CMD ["python3", "train.py"]

### Build and Run the Container

Now let's build the image and run it. We'll mount a local `output/` directory so the saved model persists after the container exits.

In [ ]:
# Build the Docker image
# The -t flag names the image; the final '.' is the build context (current directory)
!docker build -t til-ai-train:latest .

In [ ]:
import os
os.makedirs("output", exist_ok=True)
print("Created output/ directory")

In [ ]:
# Run the container with GPU access
# --rm: automatically remove the container when it exits
# --gpus all: pass all GPUs to the container
# -v: mount the local output/ dir to /workspace/output inside the container
!docker run --rm --gpus all \
    -v "$(pwd)/output:/workspace/output" \
    til-ai-train:latest

In [ ]:
# The model should now exist on your host machine
!ls -lh output/

### Running Without a GPU

If you're on a machine without an NVIDIA GPU (or without the toolkit installed), remove the `--gpus all` flag. The training script will fall back to CPU automatically:

```bash
docker run --rm \
    -v "$(pwd)/output:/workspace/output" \
    til-ai-train:latest
```

You'd also want to use a CPU-only base image to keep the image size manageable:

```dockerfile
FROM python:3.10-slim
```

CPU-only PyTorch is much lighter (~200MB vs ~3GB for the CUDA version).

## Exercise

Modify the setup above to do the following:

1. **Parameterize the number of epochs** by reading it from an environment variable (`EPOCHS`). In Docker you pass environment variables with `-e EPOCHS=50`. Inside the script, read it with `int(os.environ.get("EPOCHS", 20))`.

2. **Add a `requirements.txt`** instead of hardcoding `pip install` in the Dockerfile. Put `torch==2.2.0` there, and use `COPY requirements.txt . && RUN pip install -r requirements.txt`.

3. **Change the model architecture**: replace the 2-layer MLP with a 3-layer MLP (add an extra `nn.Linear(64, 64)` + `nn.ReLU()` before the final layer). Rebuild the image and verify you see the extra layer improving accuracy faster.

4. **(Stretch)** Add a second script `evaluate.py` that loads the saved `model.pt` and prints accuracy on a fresh set of 200 random test samples. Add it to the image and run it in a second `docker run` call (reusing the same mounted `output/` volume).

**Tip for debugging:** Add `-it` to `docker run` to get an interactive shell instead of running the default `CMD`:
```bash
docker run --rm -it --gpus all til-ai-train:latest /bin/bash
```

## 7. Inspecting Image Layers with `docker inspect`

Every Docker image is built from layers. Each `RUN`, `COPY`, or `ADD` instruction in your Dockerfile creates one layer. Understanding layer sizes helps you shrink images and diagnose why a build is larger than expected.

`docker inspect` returns a JSON object with full metadata about an image, including the layer IDs and their sizes. The cell below pulls that output, parses it, and prints each layer's compressed size in megabytes.

In [ ]:
import subprocess
import json

def inspect_image_layers(image_name: str) -> None:
    """
    Run `docker inspect` on an image and pretty-print each layer with its size.

    Docker stores layer sizes in the RootFS section (layer digests) and the
    overall image size under the `Size` key. We also pull the individual
    layer history via `docker history` for per-instruction sizes.
    """
    # --- docker inspect: overall image metadata ---
    try:
        inspect_result = subprocess.run(
            ["docker", "inspect", image_name],
            capture_output=True, text=True, check=True
        )
    except subprocess.CalledProcessError as e:
        print(f"docker inspect failed: {e.stderr.strip()}")
        print(f"Make sure the image '{image_name}' exists locally.")
        return
    except FileNotFoundError:
        print("docker not found. Is Docker installed and running?")
        return

    data = json.loads(inspect_result.stdout)
    if not data:
        print(f"No data returned for image '{image_name}'.")
        return

    info = data[0]
    total_size_mb = info.get("Size", 0) / (1024 ** 2)
    virtual_size_mb = info.get("VirtualSize", 0) / (1024 ** 2)

    print(f"Image          : {image_name}")
    print(f"ID (short)     : {info['Id'][:19]}...")
    print(f"Architecture   : {info.get('Architecture', 'unknown')}")
    print(f"OS             : {info.get('Os', 'unknown')}")
    print(f"Total size     : {total_size_mb:.1f} MB")
    print(f"Virtual size   : {virtual_size_mb:.1f} MB")
    print()

    # --- docker history: per-layer breakdown ---
    history_result = subprocess.run(
        ["docker", "history", "--no-trunc", "--format",
         '{"created_by": {{.CreatedBy}}, "size": "{{.Size}}"}', image_name],
        capture_output=True, text=True
    )

    # `docker history` with a Go template produces one JSON object per line.
    # Parse each line separately.
    print(f"{'Layer':<6}  {'Size':>10}  Created by (truncated to 60 chars)")
    print("-" * 80)
    lines = history_result.stdout.strip().split("\n")
    for idx, line in enumerate(lines):
        if not line.strip():
            continue
        # Strip the template-injected outer braces safely
        try:
            obj = json.loads(line)
            size = obj.get("size", "?")
            created_by = obj.get("created_by", "")[:60]
        except json.JSONDecodeError:
            # Fallback: docker history --format output without --no-trunc
            parts = line.split(maxsplit=1)
            size = parts[0] if parts else "?"
            created_by = parts[1][:60] if len(parts) > 1 else ""
        print(f"{idx:<6}  {size:>10}  {created_by}")

    print()
    print("Tip: large layers are usually the CUDA base image or a `pip install` step.")
    print("Use multi-stage builds (see next section) to keep the runtime image lean.")


# Inspect the image we built earlier.
# If it doesn't exist yet, this prints a helpful error instead of crashing.
inspect_image_layers("til-ai-train:latest")

## 8. Multi-Stage Builds

A single-stage Dockerfile that installs build tools (`gcc`, `cmake`, header files) carries those tools all the way into the final image. They take space and are not needed at inference time.

**Multi-stage builds** solve this by using two (or more) `FROM` stages:

1. **Builder stage**: has all the compilers, build tools, and dev headers. Compiles or installs everything.
2. **Runtime stage**: starts fresh from a lean base image and only copies the compiled artifacts from the builder.

The final image only contains what the runtime stage explicitly copies in. Everything in the builder stage is discarded.

The example below shows a realistic pattern: compile a custom C extension in the builder, then copy only the `.so` file and the Python environment into a slim runtime image.

In [ ]:
%%writefile Dockerfile.multistage
# =============================================================================
# STAGE 1: builder
# =============================================================================
# Use a full image that has compilers and build tools.
FROM python:3.10 AS builder

# Install build dependencies (e.g., for packages that compile C extensions).
# These do NOT appear in the final image.
RUN apt-get update && apt-get install -y --no-install-recommends \
    gcc \
    g++ \
    cmake \
    libffi-dev \
    && rm -rf /var/lib/apt/lists/*

WORKDIR /build

# Install Python packages into an isolated prefix so we can copy them cleanly.
COPY requirements.txt .
RUN pip install --no-cache-dir --prefix=/install -r requirements.txt

# =============================================================================
# STAGE 2: runtime
# =============================================================================
# Start fresh from a slim image (no compilers, no headers, much smaller).
FROM python:3.10-slim AS runtime

# Copy ONLY the installed Python packages from the builder stage.
# Nothing else from the builder (no gcc, no cmake, no build artifacts) is included.
COPY --from=builder /install /usr/local

WORKDIR /workspace

# Copy application code.
COPY train.py .

# Set a non-root user for security best practice.
RUN useradd --create-home appuser
USER appuser

CMD ["python3", "train.py"]

# =============================================================================
# Why this matters in numbers (approximate):
#   python:3.10           ~900 MB   (full image with build tools)
#   python:3.10-slim      ~130 MB   (just the runtime)
# Adding PyTorch CPU-only (~700 MB) to the slim image keeps total ~830 MB
# vs. ~1.6 GB if you started from the full image and never cleaned up.
# =============================================================================

## 9. Docker Compose for Multi-Container ML Setups

A real training run often needs more than one container:

- A **training container** that runs your script and writes TensorBoard event files to a shared volume.
- A **TensorBoard container** that reads those event files and serves the UI on port 6006.

Managing this with raw `docker run` commands is tedious. `docker-compose.yml` defines both containers, their shared volume, and their network in one file. You bring everything up with a single command: `docker compose up`.

The example below shows exactly this pattern.

In [ ]:
%%writefile docker-compose.yml
# docker-compose.yml for a two-container ML setup:
#   - trainer: runs the training script, writes TensorBoard logs
#   - tensorboard: reads those logs and serves the dashboard

version: "3.9"

services:

  # ---------------------------------------------------------------------------
  # Training container
  # ---------------------------------------------------------------------------
  trainer:
    build:
      context: .
      dockerfile: Dockerfile
    image: til-ai-train:latest

    # Pass all NVIDIA GPUs into this container.
    # Requires NVIDIA Container Toolkit on the host.
    deploy:
      resources:
        reservations:
          devices:
            - driver: nvidia
              count: all
              capabilities: [gpu]

    # Environment variables: tune hyperparameters without rebuilding the image.
    environment:
      - EPOCHS=100
      - BATCH_SIZE=64
      - LEARNING_RATE=1e-3

    # Mount host directories for data (read-only) and outputs (read-write).
    volumes:
      - ./data:/workspace/data:ro
      - tb_logs:/workspace/logs       # shared with tensorboard service
      - ./output:/workspace/output

    # Wait for tensorboard to be healthy before starting (optional but tidy).
    depends_on:
      - tensorboard

  # ---------------------------------------------------------------------------
  # TensorBoard container
  # ---------------------------------------------------------------------------
  tensorboard:
    image: tensorflow/tensorflow:latest
    command: tensorboard --logdir /logs --host 0.0.0.0 --port 6006

    # Expose TensorBoard on the host so you can open http://localhost:6006
    ports:
      - "6006:6006"

    volumes:
      - tb_logs:/logs   # same named volume as the trainer writes to

    # Keep TensorBoard alive even if it briefly has no log files yet.
    restart: unless-stopped

# ---------------------------------------------------------------------------
# Named volumes: managed by Docker, not tied to a specific host path.
# Both services can read and write to tb_logs simultaneously.
# ---------------------------------------------------------------------------
volumes:
  tb_logs:

# ---------------------------------------------------------------------------
# Usage:
#   docker compose up           # start both containers (attach)
#   docker compose up -d        # start in the background
#   docker compose logs -f      # stream logs from all services
#   docker compose down         # stop and remove containers
#   docker compose down -v      # also remove the named volume
# ---------------------------------------------------------------------------

## 10. Passing Secrets Safely via Environment Variables

Never bake API keys, tokens, or passwords into a Docker image. The key word is "baked in": if a secret appears in a `RUN` or `ENV` instruction, it is embedded in the image layer and will be visible to anyone who has the image, even after a later `RUN unset` command.

The correct approach: pass secrets at runtime using `-e` or `--env-file`, so they live only in the running container's environment and never touch the image filesystem.

There are three common patterns, shown below in order of increasing security.

In [ ]:
# =============================================================================
# Pattern 1: Pass a single secret inline at `docker run` time.
#
# The value never enters the image. It's only available inside the running
# container.
# =============================================================================
inline_example = """\
docker run --rm \\
  -e WANDB_API_KEY="your_actual_key_here" \\
  -e HF_TOKEN="hf_xxxxxxxxxxxx" \\
  til-ai-train:latest
"""
print("--- Pattern 1: inline -e flags ---")
print(inline_example)

# =============================================================================
# Pattern 2: Use a .env file on the host.
#
# Create a file called .env (never commit it to git!) with one KEY=VALUE
# per line, then pass it to docker run with --env-file.
# Add .env to your .gitignore and .dockerignore.
# =============================================================================
dotenv_example = """\
# Contents of .env (this file stays on your machine, never in the image):
WANDB_API_KEY=your_actual_key_here
HF_TOKEN=hf_xxxxxxxxxxxx
AWS_SECRET_ACCESS_KEY=your_aws_secret

# Then run:
docker run --rm --env-file .env til-ai-train:latest
"""
print("--- Pattern 2: --env-file .env ---")
print(dotenv_example)

# =============================================================================
# Pattern 3 (production): Docker secrets / secret mounts.
#
# For CI pipelines or Docker Swarm, use BuildKit secret mounts so the secret
# is available ONLY during the build step and never stored in any layer.
# =============================================================================
buildkit_example = """\
# In your Dockerfile, mount the secret at build time only:
#
#   RUN --mount=type=secret,id=hf_token \\
#       HF_TOKEN=$(cat /run/secrets/hf_token) \\
#       python download_model.py
#
# Build with:
#   DOCKER_BUILDKIT=1 docker build \\
#     --secret id=hf_token,src=~/.hf_token \\
#     -t my-image .
#
# The file ~/.hf_token contains just the token string.
# It is NOT copied into the image layer.
"""
print("--- Pattern 3: BuildKit secret mounts (production) ---")
print(buildkit_example)

# =============================================================================
# Reading secrets in your Python training script
# =============================================================================
import os

def get_required_env(key: str) -> str:
    """
    Read a required secret from the environment.
    Raises a clear error if it's missing rather than failing cryptically later.
    """
    value = os.environ.get(key)
    if not value:
        raise EnvironmentError(
            f"Required environment variable '{key}' is not set.\n"
            f"Pass it with: docker run -e {key}=<value> ..."
        )
    return value

# Example (would raise if WANDB_API_KEY is unset):
# api_key = get_required_env("WANDB_API_KEY")

print("get_required_env() defined. Use it in your scripts to fail fast on missing secrets.")

## 11. `.dockerignore`: Keeping Large Files Out of the Build Context

When you run `docker build .`, Docker sends everything in the current directory (the "build context") to the Docker daemon before it reads the Dockerfile. If your directory contains a 50 GB dataset or a 10 GB checkpoint folder, that transfer happens every build and makes every rebuild slow.

A `.dockerignore` file works exactly like `.gitignore`: list patterns of files and directories to exclude from the build context. The Dockerfile never sees them; they cannot be accidentally `COPY`-ed in.

In [ ]:
%%writefile .dockerignore
# =========================================================
# .dockerignore for an ML project
# =========================================================
# Everything listed here is excluded from the Docker build
# context. The Dockerfile can't COPY these files even if
# you try.
# =========================================================

# ----- Large data directories -----
data/
datasets/
raw_data/
*.csv
*.parquet
*.hdf5
*.h5

# ----- Model checkpoints and weights -----
checkpoints/
output/
*.pt
*.pth
*.ckpt
*.safetensors
*.bin
*.gguf

# ----- Jupyter artifacts -----
.ipynb_checkpoints/
*.ipynb

# ----- Python build artifacts -----
__pycache__/
*.pyc
*.pyo
*.egg-info/
dist/
build/
.eggs/

# ----- Virtual environments -----
.venv/
venv/
env/
.env/

# ----- Secrets and credentials -----
# Never let these into an image under any circumstances.
.env
*.key
*.pem
credentials.json
secrets/

# ----- Version control -----
.git/
.gitignore

# ----- Editor / OS noise -----
.DS_Store
.idea/
.vscode/
*.swp
*.swo
Thumbs.db

# ----- Logs that should stay on the host -----
runs/
logs/
wandb/
mlruns/

# =========================================================
# Pattern tip: use '!' to re-include something excluded by
# a broader rule. E.g., to exclude data/ but include a
# small sample fixture:
#
#   data/
#   !data/sample_fixture.json
# =========================================================

## Exercise

### Build a Tokenization Smoke-Test Container

Write a `Dockerfile.tokenize` that:

1. Starts from `python:3.10-slim`.
2. Installs `transformers`, `datasets`, and `accelerate` via `pip`.
3. Copies a script `tokenize_test.py` into `/workspace/`.
4. Sets `CMD ["python3", "tokenize_test.py"]`.

The script `tokenize_test.py` should:

- Load the `bert-base-uncased` tokenizer from `transformers`.
- Tokenize the sentence `"The quick brown fox jumps over the lazy dog."`.
- Print the token IDs and decoded tokens.
- Exit with code 0 if the number of tokens is between 10 and 20 (a sanity check that tokenization worked), or exit with code 1 and an error message otherwise.

**Starter code is provided below. Fill in the missing pieces.**

In [ ]:
%%writefile tokenize_test.py
"""
Tokenization smoke test.
Run inside the container to verify the environment is set up correctly.
"""
# pip install transformers datasets accelerate  (handled by the Dockerfile)
import sys

# YOUR CODE HERE: import AutoTokenizer from transformers
raise NotImplementedError("Import AutoTokenizer from transformers")

SENTENCE = "The quick brown fox jumps over the lazy dog."
MODEL_NAME = "bert-base-uncased"

def run_smoke_test():
    print(f"Loading tokenizer: {MODEL_NAME}")
    # YOUR CODE HERE: load the tokenizer using AutoTokenizer.from_pretrained
    tokenizer = None
    raise NotImplementedError("Load the tokenizer")

    print(f"Tokenizing: {SENTENCE!r}")
    # YOUR CODE HERE: tokenize the sentence and get input_ids + tokens
    encoding = None
    raise NotImplementedError("Tokenize the sentence")

    token_ids = encoding["input_ids"]
    tokens = tokenizer.convert_ids_to_tokens(token_ids)

    print(f"\nToken IDs : {token_ids}")
    print(f"Tokens    : {tokens}")
    print(f"Count     : {len(token_ids)} tokens")

    # Sanity check
    # YOUR CODE HERE: check that len(token_ids) is between 10 and 20
    raise NotImplementedError("Add the sanity check and sys.exit calls")

if __name__ == "__main__":
    run_smoke_test()

In [ ]:
%%writefile Dockerfile.tokenize
# YOUR CODE HERE: write the full Dockerfile
# Requirements:
#   - Base image: python:3.10-slim
#   - Install: transformers, datasets, accelerate
#   - Copy tokenize_test.py to /workspace/
#   - Set WORKDIR to /workspace
#   - Default CMD: python3 tokenize_test.py

raise NotImplementedError(
    "This cell uses %%writefile, so the raise won't execute in the shell.\n"
    "Replace this entire cell body with the actual Dockerfile contents.\n"
    "Remove the raise line completely once you start filling this in."
)

In [ ]:
# Once you've written both files above, build and run:
#   !docker build -t tokenize-test:latest -f Dockerfile.tokenize .
#   !docker run --rm tokenize-test:latest

# After a successful run you should see output like:
#   Loading tokenizer: bert-base-uncased
#   Tokenizing: 'The quick brown fox jumps over the lazy dog.'
#   Token IDs : [101, 1996, 4248, 2829, 4419, 14523, 2058, 1996, 13971, 3899, 1012, 102]
#   Tokens    : ['[CLS]', 'the', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog', '.', '[SEP]']
#   Count     : 12 tokens
#   Smoke test passed.

print("Build command: docker build -t tokenize-test:latest -f Dockerfile.tokenize .")
print("Run command:   docker run --rm tokenize-test:latest")